In [6]:
import psycopg2
import streamlit as st


In [80]:
def get_connection():
    try:
        conn = psycopg2.connect(
            host="localhost",
            database="Cricbuzz_Capstone",
            user="postgres",
            password="280402",
            port="5432"
        )
        return conn
    except Exception as e:
        st.error(f"PostgreSQL connection failed: {e}")
        return None

In [ ]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "d1d1206e78mshb418f1da663697ap109e25jsn86275d1f5944",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
}

conn.request("GET", "/matches/v1/live", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


In [ ]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
}

conn.request("GET", "/stats/v1/topstats/0?statsType=mostWickets", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


In [ ]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/teams/v1/2/players", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


International Cricket Teams


In [ ]:
import http.client
import json

def fetch_and_insert_teams():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    # API connection
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
         'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request("GET", "/teams/v1/international", headers=headers)

    res = conn.getresponse()
    data = json.loads(res.read().decode("utf-8"))

    current_category = None

    for item in data["list"]:

        # Handle category rows like "Test Teams"
        if "teamId" not in item:
            current_category = item.get("teamName")
            continue

        team_id = item["teamId"]
        team_name = item["teamName"]
        short_name = item.get("teamSName")

        try:
            cursor.execute("""
                INSERT INTO teams (team_id, team_name, short_name, category)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT (team_id) DO NOTHING;
            """, (team_id, team_name, short_name, current_category))

        except Exception as e:
            print(f"Error inserting team {team_name}: {e}")

    conn_db.commit()
    cursor.close()
    conn_db.close()

    print("✅ Teams inserted successfully!")


# run
fetch_and_insert_teams()

✅ Teams inserted successfully!


In [ ]:
import http.client
import json
import time
import psycopg2


# ✅ Your DB connection
def get_connection():
    try:
        conn = psycopg2.connect(
            host="localhost",
            database="Cricbuzz_Capstone",
            user="postgres",
            password="280402",
            port="5432"
        )
        return conn
    except Exception as e:
        print(f"PostgreSQL connection failed: {e}")
        return None


def fetch_and_insert_players():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    # ✅ Get all team_ids
    cursor.execute("SELECT team_id FROM teams;")
    teams = cursor.fetchall()

    headers = {
           'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
            'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    for (team_id,) in teams:
        print(f"\n📡 Fetching players for team {team_id}...")

        try:
            #new connection per request
            conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

            conn_api.request("GET", f"/teams/v1/{team_id}/players", headers=headers)
            res = conn_api.getresponse()

            # No data case (normal)
            if res.status == 204:
                print(f"No players for team {team_id} (204)")
                continue

            #Any other failure
            if res.status != 200:
                print(f"Failed for team {team_id}, status: {res.status}")
                continue

            data = json.loads(res.read().decode("utf-8"))

            current_role = None

            for item in data.get("player", []):

                # Handle role headers
                if "id" not in item:
                    current_role = item.get("name")
                    continue

                player_id = int(item["id"])
                player_name = item["name"]
                batting_style = item.get("battingStyle")
                bowling_style = item.get("bowlingStyle")

                cursor.execute("""
                    INSERT INTO players (
                        player_id,
                        player_name,
                        team_id,
                        batting_style,
                        bowling_style,
                        role
                    )
                    VALUES (%s, %s, %s, %s, %s, %s)
                    ON CONFLICT (player_id) DO NOTHING;
                """, (
                    player_id,
                    player_name,
                    team_id,
                    batting_style,
                    bowling_style,
                    current_role
                ))

            conn_db.commit()
            print(f"Inserted players for team {team_id}")

            time.sleep(1)

        except Exception as e:
            print(f"Error for team {team_id}: {e}")

    cursor.close()
    conn_db.close()

    print("\n All players inserted successfully!")


# ▶️ RUN
fetch_and_insert_players()


📡 Fetching players for team 2...
✅ Inserted players for team 2

📡 Fetching players for team 96...
✅ Inserted players for team 96

📡 Fetching players for team 27...
✅ Inserted players for team 27

📡 Fetching players for team 3...
✅ Inserted players for team 3

📡 Fetching players for team 4...
✅ Inserted players for team 4

📡 Fetching players for team 5...
✅ Inserted players for team 5

📡 Fetching players for team 6...
✅ Inserted players for team 6

📡 Fetching players for team 9...
✅ Inserted players for team 9

📡 Fetching players for team 10...
✅ Inserted players for team 10

📡 Fetching players for team 11...
✅ Inserted players for team 11

📡 Fetching players for team 12...
✅ Inserted players for team 12

📡 Fetching players for team 13...
✅ Inserted players for team 13

📡 Fetching players for team 71...
⚠️ No players for team 71 (204)

📡 Fetching players for team 72...
✅ Inserted players for team 72

📡 Fetching players for team 77...
⚠️ No players for team 77 (204)

📡 Fetching players 

In [ ]:
import http.client
import json

def fetch_and_insert_matches():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    headers = {
         'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")
    conn_api.request("GET", "/matches/v1/recent", headers=headers)
    res = conn_api.getresponse()

    data = json.loads(res.read().decode("utf-8"))

    for type_match in data.get("typeMatches", []):
        for series in type_match.get("seriesMatches", []):

            if "seriesAdWrapper" not in series:
                continue

            series_data = series["seriesAdWrapper"]
            series_id = series_data.get("seriesId")

            for match in series_data.get("matches", []):

                match_info = match.get("matchInfo", {})
                match_id = match_info.get("matchId")

                team1 = match_info.get("team1", {})
                team2 = match_info.get("team2", {})

                team1_id = team1.get("teamId")
                team2_id = team2.get("teamId")

                team1_name = team1.get("teamName")
                team2_name = team2.get("teamName")

                match_desc = match_info.get("matchDesc")
                match_format = match_info.get("matchFormat")
                status = match_info.get("status")
                state = match_info.get("state")

                start_date = match_info.get("startDate")
                end_date = match_info.get("endDate")

                venue = match_info.get("venueInfo", {}).get("ground")

                try:
                    #AUTO-INSERT missing teams (CRITICAL FIX)
                    cursor.execute("""
                        INSERT INTO teams (team_id, team_name)
                        VALUES (%s, %s)
                        ON CONFLICT (team_id) DO NOTHING;
                    """, (team1_id, team1_name))

                    cursor.execute("""
                        INSERT INTO teams (team_id, team_name)
                        VALUES (%s, %s)
                        ON CONFLICT (team_id) DO NOTHING;
                    """, (team2_id, team2_name))

                    #derive winner
                    winner_id = None
                    if state == "Complete" and status:
                        if team1_name and team1_name in status:
                            winner_id = team1_id
                        elif team2_name and team2_name in status:
                            winner_id = team2_id

                    #insert match
                    cursor.execute("""
                        INSERT INTO matches (
                            match_id, series_id, match_desc, match_format,
                            start_date, end_date, status,
                            team1_id, team2_id, winner_team_id, venue
                        )
                        VALUES (%s, %s, %s, %s,
                                to_timestamp(%s/1000.0),
                                to_timestamp(%s/1000.0),
                                %s,
                                %s, %s, %s, %s)
                        ON CONFLICT (match_id) DO NOTHING;
                    """, (
                        match_id,
                        series_id,
                        match_desc,
                        match_format,
                        start_date,
                        end_date,
                        status,
                        team1_id,
                        team2_id,
                        winner_id,
                        venue
                    ))

                except Exception as e:
                    print(f"Error inserting match {match_id}: {e}")
                    conn_db.rollback()

    conn_db.commit()
    cursor.close()
    conn_db.close()

    print("Matches inserted successfully!")

fetch_and_insert_matches()

✅ Matches inserted successfully!


Stadium Venues Insertion


In [237]:
import http.client
import json
import pandas as pd

In [ ]:
def fetch_venue_data(venue_id):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
     'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    endpoint = f"/venues/v1/{venue_id}"
    conn.request("GET", endpoint, headers=headers)

    res = conn.getresponse()
    data = res.read()

    venue = json.loads(data.decode("utf-8"))
    return venue

In [239]:
venue = fetch_venue_data(45)
venue

{'ground': 'Basin Reserve',
 'city': 'Wellington',
 'country': 'New Zealand',
 'timezone': '+13:00',
 'capacity': '11,600',
 'ends': 'Vance Stand End, Scoreboard End',
 'homeTeam': 'Wellington',
 'imageUrl': 'http://i.cricketcb.com/i/stats/fth/540x303/venue/images/45.jpg',
 'imageId': '189174'}

In [ ]:
import http.client
import json

def fetch_matches():
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
         'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    # This endpoint gives list of matches
    conn.request("GET", "/matches/v1/recent", headers=headers)

    res = conn.getresponse()
    data = res.read()

    return json.loads(data.decode("utf-8"))

In [241]:
def extract_venue_ids(match_data):
    venue_ids = set()

    for match_type in match_data.get("typeMatches", []):
        for series in match_type.get("seriesMatches", []):
            for match in series.get("seriesAdWrapper", {}).get("matches", []):
                
                info = match.get("matchInfo", {})
                venue = info.get("venueInfo", {})

                vid = venue.get("id")
                if vid:
                    venue_ids.add(vid)

    return list(venue_ids)

In [242]:
matches = fetch_matches()
venue_ids = extract_venue_ids(matches)

venue_ids

[26,
 27,
 31,
 418,
 45,
 1461,
 1864,
 1482,
 1871,
 81,
 851,
 1438295,
 88,
 96,
 100,
 485,
 1893,
 363,
 371,
 116,
 122,
 380]

In [ ]:
import re

def get_venue(venue_id, venue):
    raw_capacity = venue.get("capacity", "")

    # Extract only numbers using regex
    numbers = re.findall(r"\d+", raw_capacity.replace(",", ""))

    capacity = int(numbers[0]) if numbers else None

    return {
        "venue_id": venue_id,
        "venue_name": venue.get("ground"),
        "city": venue.get("city"),
        "country": venue.get("country"),
        "timezone": venue.get("timezone"),
        "capacity": capacity,
        "ends": venue.get("ends"),
        "home_team": venue.get("homeTeam")
    }

In [244]:
def insert_venue(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO venues (
            venue_id, venue_name, city, country, timezone,
            capacity, ends, home_team
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (venue_id) DO NOTHING;
    """

    cursor.execute(query, (
        data["venue_id"],
        data["venue_name"],
        data["city"],
        data["country"],
        data["timezone"],
        data["capacity"],
        data["ends"],
        data["home_team"]
    ))

    conn.commit()
    cursor.close()
    conn.close()

In [ ]:
def fetch_store_from_matches():
    matches = fetch_matches()
    venue_ids = extract_venue_ids(matches)

    print(f"Found {len(venue_ids)} venues")

    for vid in venue_ids:
        try:
            venue = fetch_venue_data(vid)
            processed = get_venue(vid, venue)
            insert_venue(processed)

            print(f"Stored venue {vid}")

        except Exception as e:
            print(f"Skipped {vid}: {e}")

Extracting Recent Matches


In [ ]:
def extract_matches_and_teams(data):
    matches = []
    teams = {}

    for match_type in data.get("typeMatches", []):
        for series in match_type.get("seriesMatches", []):
            wrapper = series.get("seriesAdWrapper")
            if not wrapper:
                continue

            for match in wrapper.get("matches", []):
                info = match.get("matchInfo", {})

                # Extract Teams
                t1 = info.get("team1", {})
                t2 = info.get("team2", {})

                if t1:
                    teams[t1["teamId"]] = {
                        "team_id": t1["teamId"],
                        "team_name": t1["teamName"],
                        "short_name": t1["teamSName"]
                    }

                if t2:
                    teams[t2["teamId"]] = {
                        "team_id": t2["teamId"],
                        "team_name": t2["teamName"],
                        "short_name": t2["teamSName"]
                    }

                # Convert timestamp
                from datetime import datetime
                start_ts = info.get("startDate")
                start_date = datetime.fromtimestamp(int(start_ts)/1000) if start_ts else None

                # Extract Match
                matches.append({
                    "match_id": info.get("matchId"),
                    "series_id": info.get("seriesId"),
                    "match_desc": info.get("matchDesc"),
                    "match_format": info.get("matchFormat"),
                    "start_date": start_date,
                    "team1_id": t1.get("teamId"),
                    "team2_id": t2.get("teamId"),
                    "venue_id": info.get("venueInfo", {}).get("id")
                })

    return matches, list(teams.values())

In [247]:
data = fetch_matches()

matches, teams = extract_matches_and_teams(data)

print("Matches:", len(matches))
print("Teams:", len(teams))

Matches: 41
Teams: 51


In [248]:
def insert_teams(teams):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO teams (team_id, team_name, short_name)
    VALUES (%s, %s, %s)
    ON CONFLICT (team_id) DO NOTHING;
    """

    for t in teams:
        cursor.execute(query, (
            t["team_id"],
            t["team_name"],
            t["short_name"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [249]:
from datetime import datetime

def insert_matches(matches):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO matches (
        match_id, series_id, match_desc, match_format,
        start_date, end_date, status,
        team1_id, team2_id, venue_id
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (match_id) DO NOTHING;
    """

    for m in matches:
        cursor.execute(query, (
            m["match_id"],
            m["series_id"],
            m["match_desc"],
            m["match_format"],
            m["start_date"],
            m.get("end_date"),
            m.get("status"),
            m["team1_id"],
            m["team2_id"],
            m["venue_id"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [250]:
def extract_venue_ids_from_matches(matches):
    return list(set([m["venue_id"] for m in matches if m["venue_id"]]))

In [ ]:
def insert_missing_venues(matches):
    venue_ids = extract_venue_ids_from_matches(matches)

    print(f"Found {len(venue_ids)} venues")

    for vid in venue_ids:
        try:
            venue = fetch_venue_data(vid)

            if not venue or "ground" not in venue:
                continue

            processed = get_venue(vid, venue)
            insert_venue(processed)

            print(f"Venue {vid} inserted")

        except Exception as e:
            print(f"Skipped venue {vid}: {e}")

In [ ]:
data = fetch_matches()

matches, teams = extract_matches_and_teams(data)

insert_missing_venues(matches)   
insert_teams(teams)              
insert_matches(matches)          

print("Full pipeline executed")

Found 22 venues
✅ Venue 26 inserted
✅ Venue 27 inserted
✅ Venue 31 inserted
✅ Venue 418 inserted
✅ Venue 45 inserted
✅ Venue 1461 inserted
✅ Venue 1864 inserted
✅ Venue 1482 inserted
✅ Venue 1871 inserted
✅ Venue 81 inserted
✅ Venue 851 inserted
✅ Venue 1438295 inserted
✅ Venue 88 inserted
✅ Venue 96 inserted
✅ Venue 100 inserted
✅ Venue 485 inserted
✅ Venue 1893 inserted
✅ Venue 363 inserted
✅ Venue 371 inserted
✅ Venue 116 inserted
✅ Venue 122 inserted
✅ Venue 380 inserted
✅ Full pipeline executed


Fetching data of top ODI run scorers

In [ ]:
def fetch_top_run_scorers():
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
          'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request(
        "GET",
        "/stats/v1/topstats/0?statsType=mostRuns&matchType=2",
        headers=headers
    )

    res = conn.getresponse()
    data = res.read()
    print(json.loads(data.decode("utf-8")))
    return json.loads(data.decode("utf-8"))

In [254]:
players = process_run_scorers(data)
print(players[:3])

[]


In [255]:
def process_run_scorers(data):
    players = []

    for row in data.get("values", []):
        vals = row["values"]

        players.append({
            "player_name": vals[1],
            "format_type": "ODI",
            "matches": int(vals[2]),
            "innings": int(vals[3]),
            "runs": int(vals[4]),
            "batting_average": float(vals[5]),
            "rank_position": int(vals[0])
        })

    return players

In [256]:
def insert_run_scorers(players):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
    INSERT INTO top_run_scorers (
        player_name, format_type, matches, innings,
        runs, batting_average, rank_position
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (player_name, format_type) DO UPDATE SET
        runs = EXCLUDED.runs,
        batting_average = EXCLUDED.batting_average,
        rank_position = EXCLUDED.rank_position;
    """

    for p in players:
        cursor.execute(query, (
            p["player_name"],
            p["format_type"],
            p["matches"],
            p["innings"],
            p["runs"],
            p["batting_average"],
            p["rank_position"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [ ]:
data = fetch_top_run_scorers()
players = process_run_scorers(data)

insert_run_scorers(players)

print("Run scorers inserted")

✅ Run scorers inserted


In [258]:
data = fetch_top_run_scorers()
print(data)

{'filter': {'matchtype': [{'matchTypeId': '1', 'matchTypeDesc': 'test'}, {'matchTypeId': '2', 'matchTypeDesc': 'odi'}, {'matchTypeId': '3', 'matchTypeDesc': 't20i'}], 'team': [{'id': '2', 'teamShortName': 'IND'}, {'id': '27', 'teamShortName': 'IRE'}, {'id': '3', 'teamShortName': 'PAK'}, {'id': '4', 'teamShortName': 'AUS'}, {'id': '5', 'teamShortName': 'SL'}, {'id': '6', 'teamShortName': 'BAN'}, {'id': '9', 'teamShortName': 'ENG'}, {'id': '10', 'teamShortName': 'WI'}, {'id': '11', 'teamShortName': 'RSA'}, {'id': '12', 'teamShortName': 'ZIM'}, {'id': '13', 'teamShortName': 'NZ'}, {'id': '96', 'teamShortName': 'AFG'}], 'selectedMatchType': 'odi'}, 'headers': ['Batter', 'M', 'I', 'R', 'Avg'], 'values': [{'values': ['25', 'Tendulkar', '463', '452', '18426', '44.83']}, {'values': ['1413', 'Virat Kohli', '311', '299', '14797', '58.72']}, {'values': ['104', 'Sangakkara', '404', '380', '14234', '41.99']}, {'values': ['38', 'Ricky Ponting', '375', '365', '13704', '42.04']}, {'values': ['102', 'S

In [259]:
df = pd.read_sql("SELECT COUNT(*) FROM top_run_scorers;", get_connection())
df

C:\Users\hp\AppData\Local\Temp\ipykernel_34636\3334193213.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT COUNT(*) FROM top_run_scorers;", get_connection())


,count
0,20


Highest score in each format


In [290]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/stats/v1/topstats?statsType=highestScore&formatType=t20", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"filter":{"matchtype":[{"matchTypeId":"1","matchTypeDesc":"test"},{"matchTypeId":"2","matchTypeDesc":"odi"},{"matchTypeId":"3","matchTypeDesc":"t20i"}],"team":[{"id":"2","teamShortName":"IND"},{"id":"27","teamShortName":"IRE"},{"id":"3","teamShortName":"PAK"},{"id":"4","teamShortName":"AUS"},{"id":"5","teamShortName":"SL"},{"id":"6","teamShortName":"BAN"},{"id":"9","teamShortName":"ENG"},{"id":"10","teamShortName":"WI"},{"id":"11","teamShortName":"RSA"},{"id":"12","teamShortName":"ZIM"},{"id":"13","teamShortName":"NZ"},{"id":"96","teamShortName":"AFG"}],"selectedMatchType":"test"},"headers":["Batter","HS","Balls","SR","Vs"],"values":[{"values":["240","B Lara","400","582","68.73","ENG"]},{"values":["157","Matthew Hayden","380","437","86.96","ZIM"]},{"values":["240","B Lara","375","538","69.70","ENG"]},{"values":["101","Mahela","374","572","65.38","RSA"]},{"values":["11200","Mulder","367","334","109.88","ZIM"]},{"values":["4160","G Sobers","365","","","PAK"]},{"values":["5141","S Hutton

In [ ]:
import http.client
import json


def get_highest_score_header():
    # API connection
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
        'Content-Type': "application/json"
    }

    try:
        # API request
        conn.request("GET", "/stats/v1/topstats", headers=headers)

        res = conn.getresponse()

        print(f"Status: {res.status}")

        if res.status != 200:
            print("API failed")
            return

        # Read + parse JSON
        data = json.loads(res.read().decode("utf-8"))

        #Extract "Highest Scores"
        for category in data.get("statsTypesList", []):
            for stat in category.get("types", []):
                if stat.get("value") == "highestScore":
                    print("✅ Header Found:", stat.get("header"))
                    return stat.get("header")

        print("highestScore not found")

    except Exception as e:
        print(f"Error: {e}")

    finally:
        conn.close()


# RUN
if __name__ == "__main__":
    print("🚀 Running script...")
    result = get_highest_score_header()
    print("Final Output:", result)

{"statsTypesList":[{"types":[{"value":"mostRuns","header":"Most Runs","category":"Batting"},{"value":"highestScore","header":"Highest Scores","category":"Batting"},{"value":"highestAvg","header":"Best Batting Average","category":"Batting"},{"value":"highestSr","header":"Best Batting Strike Rate","category":"Batting"},{"value":"mostHundreds","header":"Most Hundreds","category":"Batting"},{"value":"mostFifties","header":"Most Fifties","category":"Batting"},{"value":"mostFours","header":"Most Fours","category":"Batting"},{"value":"mostSixes","header":"Most Sixes","category":"Batting"},{"value":"mostNineties","header":"Most Nineties","category":"Batting"}],"category":"Batting"},{"types":[{"value":"mostWickets","header":"Most Wickets","category":"Bowling"},{"value":"lowestAvg","header":"Best Bowling Average","category":"Bowling"},{"value":"bestBowlingInnings","header":"Best Bowling ","category":"Bowling"},{"value":"mostFiveWickets","header":"Most 5 Wickets Haul","category":"Bowling"},{"valu

In [ ]:
def insert_highest_scores():
    import http.client
    import json

    print("FUNCTION STARTED")

    conn_db = get_connection()
    if not conn_db:
        print("DB connection failed")
        return

    cursor = conn_db.cursor()

    headers = {
        'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    formats = ["test", "odi", "t20i"]

    for format_type in formats:
        print(f"\n Fetching {format_type.upper()} data...")

        try:
            conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

            conn_api.request(
                "GET",
                f"/stats/v1/topstats?statsType=highestScore&formatType={format_type}",
                headers=headers
            )

            res = conn_api.getresponse()
            print(f" STATUS: {res.status}")

            if res.status != 200:
                print(f" API failed for {format_type}")
                continue

            data = json.loads(res.read().decode("utf-8"))

            inserted_count = 0

            for item in data.get("values", []):

                try:
                    row = item.get("values", [])

                    if not row or len(row) < 3:
                        continue

                    player_name = row[1]

                    # safe score parsing
                    try:
                        highest_score = int(row[2])
                    except:
                        continue

                    balls = int(row[3]) if len(row) > 3 and row[3].isdigit() else None

                    strike_rate = None
                    if len(row) > 4 and row[4]:
                        try:
                            strike_rate = float(row[4])
                        except:
                            pass

                    opponent = row[5] if len(row) > 5 else None

                    cursor.execute("""
                        INSERT INTO highest_scores 
                        (player_name, format_type, highest_score, balls, strike_rate, opponent)
                        VALUES (%s, %s, %s, %s, %s, %s)
                        ON CONFLICT DO NOTHING;
                    """, (
                        player_name,
                        format_type.upper(),
                        highest_score,
                        balls,
                        strike_rate,
                        opponent
                    ))

                    inserted_count += 1

                except Exception as e:
                    print(f"Row error: {row} → {e}")
                    conn_db.rollback()

            conn_db.commit()
            print(f"Inserted {inserted_count} rows for {format_type}")

        except Exception as e:
            print(f"ERROR for {format_type}: {e}")

    cursor.close()
    conn_db.close()
    print("\n DONE inserting highest scores!")

Series from 2024

In [ ]:
import http.client
import json


def insert_series():
    conn_db = get_connection()
    if not conn_db:
        return

    cursor = conn_db.cursor()

    conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
          'x-rapidapi-key': "274cf7b009msh198245adc41e439p1f3633jsna216dc087842",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    try:
        conn_api.request("GET", "/series/v1/archives/international", headers=headers)

        res = conn_api.getresponse()

        print("STATUS:", res.status)

        if res.status != 200:
            print("API failed")
            return

        data = json.loads(res.read().decode("utf-8"))

        inserted = 0

        for year_block in data.get("seriesMapProto", []):
            year = year_block.get("date")

            for series in year_block.get("series", []):

                try:
                    series_id = series.get("id")
                    series_name = series.get("name")

                    start_dt = series.get("startDt")
                    end_dt = series.get("endDt")

                    cursor.execute("""
                        INSERT INTO series (
                            series_id,
                            series_name,
                            start_date,
                            end_date
                        )
                        VALUES (%s, %s,
                                to_timestamp(%s/1000.0),
                                to_timestamp(%s/1000.0))
                        ON CONFLICT DO NOTHING;
                    """, (
                        series_id,
                        series_name,
                        start_dt,
                        end_dt
                    ))

                    inserted += 1

                except Exception as e:
                    print(f"Error: {e}")
                    conn_db.rollback()

        conn_db.commit()
        print(f"Inserted {inserted} series")

    except Exception as e:
        print(f"API error: {e}")

    finally:
        cursor.close()
        conn_db.close()


#RUN
insert_series()

STATUS: 200
✅ Inserted 20 series


Most wickets and Runs

In [360]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
      'x-rapidapi-key': "cd3e8d5088mshfe5a93d6d4454aap1cf400jsn26d789da781e",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/stats/v1/player/9647/batting", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


In [353]:
def get_all_player_ids():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT player_id FROM players")
    ids = [row[0] for row in cursor.fetchall()]

    cursor.close()
    conn.close()
    return ids

In [354]:
def get_existing_player_ids():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT player_id FROM player_stats")
    ids = {row[0] for row in cursor.fetchall()}

    cursor.close()
    conn.close()
    return ids

In [355]:
import json

def safe_json_load(data):
    try:
        text = data.decode("utf-8").strip()
        if not text:
            return None
        return json.loads(text)
    except:
        return None

In [356]:
import time
import http.client

def make_api_request(conn, url, headers, player_id):
    retries = 3

    for attempt in range(retries):
        conn.request("GET", url, headers=headers)
        res = conn.getresponse()
        data = res.read()

        if res.status == 200:
            return safe_json_load(data)

        elif res.status == 429:
            wait = 2 ** attempt
            print(f"429 for {player_id} → retry in {wait}s")
            time.sleep(wait)

        else:
            print(f"API failed for {player_id}, status {res.status}")
            return None

    return None

In [357]:
def process_player_stats(player_id):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
           'x-rapidapi-key': "cd3e8d5088mshfe5a93d6d4454aap1cf400jsn26d789da781e",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    try:
        # ---------- Batting ----------
        batting_data = make_api_request(
            conn,
            f"/stats/v1/player/{player_id}/batting",
            headers,
            player_id
        )

        if not batting_data:
            return None

        # ---------- Bowling ----------
        bowling_data = make_api_request(
            conn,
            f"/stats/v1/player/{player_id}/bowling",
            headers,
            player_id
        )

        if not bowling_data:
            return None

        # ---------- Extract Batting ----------
        total_runs = 0
        total_matches = 0
        batting_avg = 0
        strike_rate = 0

        for row in batting_data.get("values", []):
            label = row["values"][0]

            if label == "Runs":
                total_runs = sum(int(x) for x in row["values"][1:] if x.isdigit())

            elif label == "Matches":
                total_matches = sum(int(x) for x in row["values"][1:] if x.isdigit())

            elif label == "Average":
                vals = [float(x) for x in row["values"][1:] if x.replace('.', '', 1).isdigit()]
                batting_avg = sum(vals)/len(vals) if vals else 0

            elif label == "SR":
                vals = [float(x) for x in row["values"][1:] if x.replace('.', '', 1).isdigit()]
                strike_rate = sum(vals)/len(vals) if vals else 0


        # ---------- Extract Bowling ----------
        total_wickets = 0
        bowling_avg = 0
        economy = 0

        for row in bowling_data.get("values", []):
            label = row["values"][0]

            if label == "Wickets":
                total_wickets = sum(int(x) for x in row["values"][1:] if x.isdigit())

            elif label == "Avg":
                vals = [float(x) for x in row["values"][1:] if x.replace('.', '', 1).isdigit()]
                bowling_avg = sum(vals)/len(vals) if vals else 0

            elif label == "Eco":
                vals = [float(x) for x in row["values"][1:] if x.replace('.', '', 1).isdigit()]
                economy = sum(vals)/len(vals) if vals else 0


        return {
            "player_id": player_id,
            "total_matches": total_matches,
            "total_runs": total_runs,
            "batting_average": batting_avg,
            "strike_rate": strike_rate,
            "total_wickets": total_wickets,
            "bowling_average": bowling_avg,
            "economy": economy
        }

    except Exception as e:
        print(f"Error for player {player_id}: {e}")
        return None

In [358]:
def insert_player_stats(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO player_stats (
            player_id, total_matches, total_runs,
            batting_average, strike_rate,
            total_wickets, bowling_average, economy
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (player_id) DO UPDATE
        SET total_runs = EXCLUDED.total_runs,
            total_wickets = EXCLUDED.total_wickets,
            batting_average = EXCLUDED.batting_average,
            strike_rate = EXCLUDED.strike_rate,
            bowling_average = EXCLUDED.bowling_average,
            economy = EXCLUDED.economy;
    """

    cursor.execute(query, (
        data["player_id"],
        data["total_matches"],
        data["total_runs"],
        data["batting_average"],
        data["strike_rate"],
        data["total_wickets"],
        data["bowling_average"],
        data["economy"]
    ))

    conn.commit()
    cursor.close()
    conn.close()

In [365]:
import http.client
import json

def get_matches_by_format():
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
         'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request("GET", f"/stats/v1/player/576/batting", headers=headers)

    res = conn.getresponse()
    data = res.read()

    batting_data = json.loads(data.decode("utf-8"))

    formats = batting_data.get("headers", [])[1:]  # ['Test','ODI','T20','IPL']

    matches_data = {}

    for row in batting_data.get("values", []):
        if row["values"][0] == "Matches":

            for i, fmt in enumerate(formats):
                val = row["values"][i+1]

                matches_data[fmt.lower()] = int(val) if val.isdigit() else 0

    return {
        "test": matches_data.get("test", 0),
        "odi": matches_data.get("odi", 0),
        "t20": matches_data.get("t20", 0)
    }

In [367]:
get_matches_by_format()

{'test': 67, 'odi': 282, 't20': 159}

In [ ]:
import http.client
import json
import time

# GET LIMITED PLAYER IDS
def get_limited_player_ids(limit=20):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(f"""
        SELECT player_id
        FROM players
        WHERE player_id IS NOT NULL
        LIMIT {limit};
    """)

    ids = [row[0] for row in cursor.fetchall()]

    cursor.close()
    conn.close()
    return ids


# PROCESS PLAYER (EXTRACT)
def process_player_stats(player_id):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    conn.request("GET", f"/stats/v1/player/{player_id}/batting", headers=headers)

    res = conn.getresponse()
    data = res.read()

    try:
        batting_data = json.loads(data.decode("utf-8"))
    except:
        return []

    formats = batting_data.get("headers", [])[1:]

    result = []
    valid_formats = 0

    for i, fmt in enumerate(formats):

        if fmt not in ["Test", "ODI", "T20"]:
            continue

        matches = 0
        runs = 0
        avg = 0

        for row in batting_data.get("values", []):
            label = row["values"][0]
            val = row["values"][i+1]

            if label == "Matches":
                matches = int(val) if val.isdigit() else 0

            elif label == "Runs":
                runs = int(val) if val.isdigit() else 0

            elif label == "Average":
                avg = float(val) if val.replace('.', '', 1).isdigit() else 0

        if matches > 0:
            valid_formats += 1

            result.append({
                "player_id": player_id,
                "format_type": fmt,
                "matches": matches,
                "runs": runs,
                "batting_average": avg
            })

    if valid_formats < 2:
        return []

    return result


# INSERT INTO TABLE
def insert_player_format_stats(data_list):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO player_format_stats (
            player_id, format_type, matches, runs, batting_average
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (player_id, format_type) DO UPDATE
        SET runs = EXCLUDED.runs,
            batting_average = EXCLUDED.batting_average;
    """

    for data in data_list:
        cursor.execute(query, (
            data["player_id"],
            data["format_type"],
            data["matches"],
            data["runs"],
            data["batting_average"]
        ))

    conn.commit()
    cursor.close()
    conn.close()


# MAIN RUNNER
def run_q11_pipeline():
    player_ids = get_limited_player_ids(20)

    for pid in player_ids:
        stats = process_player_stats(pid)

        if stats:
            insert_player_format_stats(stats)

        time.sleep(2)  

    print("Data stored successfully!")


# RUN
run_q11_pipeline()

✅ Data stored successfully!


Partnerships of batsman


In [376]:
def get_match_ids(limit=5):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute(f"""
        SELECT match_id
        FROM matches
        WHERE match_id IS NOT NULL
        LIMIT {limit};
    """)

    match_ids = [row[0] for row in cursor.fetchall()]

    cursor.close()
    conn.close()

    return match_ids

In [ ]:
import http.client
import json
import time

def extract_partnerships(match_ids):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
         'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    results = []

    for match_id in match_ids:
        try:
            conn.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
            res = conn.getresponse()
            data = res.read()

            match_data = json.loads(data.decode("utf-8"))

            for innings in match_data.get("scorecard", []):

                innings_id = innings.get("inningsid")

                partnerships = innings.get("partnership", {}).get("partnership", [])

                for p in partnerships:
                    runs = p.get("totalruns", 0)

                    if runs >= 50:
                        results.append({
                            "match_id": match_id,
                            "innings": innings_id,
                            "player1": p.get("bat1name"),
                            "player2": p.get("bat2name"),
                            "partnership_runs": runs
                        })

        except Exception as e:
            print(f"Error in match {match_id}: {e}")

        time.sleep(2)

    return results

In [382]:
def insert_partnerships(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO partnerships (
            match_id, innings, player1, player2, partnership_runs
        )
        VALUES (%s, %s, %s, %s, %s);
    """

    for row in data:
        cursor.execute(query, (
            row["match_id"],
            row["innings"],
            row["player1"],
            row["player2"],
            row["partnership_runs"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [ ]:
match_ids = get_match_ids(5)

data = extract_partnerships(match_ids)

print(data)

insert_partnerships(data)

[{'match_id': 151169, 'innings': 1, 'player1': 'Shantanu Kaveri', 'player2': 'Ankush Singh', 'partnership_runs': 69}, {'match_id': 151158, 'innings': 2, 'player1': 'Luiz Muller', 'player2': 'Luiz Morais', 'partnership_runs': 66}, {'match_id': 151147, 'innings': 1, 'player1': 'Kashigoud Patil', 'player2': 'Shantanu Kaveri', 'partnership_runs': 64}, {'match_id': 151147, 'innings': 1, 'player1': 'Amir Butt', 'player2': 'Dhananjaya Panda', 'partnership_runs': 135}, {'match_id': 151141, 'innings': 1, 'player1': 'Kashigoud Patil', 'player2': 'Gurpreet Singh', 'partnership_runs': 64}, {'match_id': 151141, 'innings': 2, 'player1': 'Yasar Haroon', 'player2': 'Luiz Morais', 'partnership_runs': 81}]


In [453]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/mcenter/v1/149640/scard", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"scorecard":[{"inningsid":1,"batsman":[{"id":8271,"balls":7,"runs":6,"fours":1,"sixes":0,"strkrate":"85.71","name":"Sanju Samson","nickname":"Sanju Samson","iscaptain":false,"iskeeper":true,"outdec":"b Nandre Burger","videotype":"","videourl":"","videoid":0,"planid":0,"imageid":0,"premiumvideourl":"","iscbplusfree":false,"ispremiumfree":false,"inmatchchange":"","isoverseas":false,"playingxichange":""},{"id":11813,"balls":11,"runs":6,"fours":1,"sixes":0,"strkrate":"54.55","name":"Ruturaj Gaikwad","nickname":"Ruturaj Gaikwad","iscaptain":true,"iskeeper":false,"outdec":"b Jofra Archer","videotype":"","videourl":"","videoid":0,"planid":0,"imageid":0,"premiumvideourl":"","iscbplusfree":false,"ispremiumfree":false,"inmatchchange":"","isoverseas":false,"playingxichange":""},{"id":1431163,"balls":1,"runs":0,"fours":0,"sixes":0,"strkrate":"0","name":"Ayush Mhatre","nickname":"Ayush Mhatre","iscaptain":false,"iskeeper":false,"outdec":"c Dhruv Jurel b Nandre Burger","videotype":"","videourl":"",

Bowling Performance at different venue

In [437]:
import http.client
import json
import psycopg2
import time
import streamlit as st

In [447]:
def get_match_ids(conn):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT match_id
        FROM matches_v2
        WHERE match_id NOT IN (
            SELECT DISTINCT match_id FROM player_match_performance
        )
        LIMIT 15;
    """)
    rows = cursor.fetchall()
    return [r[0] for r in rows]

In [ ]:
def fetch_match_data(match_id):
    try:
        conn = http.client.HTTPSConnection(
            "cricbuzz-cricket.p.rapidapi.com",
            timeout=10
        )

        headers = {
             'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
            'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
            'Content-Type': "application/json"
        }

        conn.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
        res = conn.getresponse()

        if res.status != 200:
            print(f"Failed {match_id} - Status {res.status}")
            return None

        data = json.loads(res.read().decode("utf-8"))

        print(f"\nRAW RESPONSE for {match_id}:")
        print(data)

        return data

    except Exception as e:
        print(f"Failed {match_id}: {e}")
        return None

In [440]:
def store_match_info(cursor, data, match_id):

    status = data.get("status", "")

    # simple parsing
    win_margin = None
    win_type = None

    if "runs" in status:
        win_type = "runs"
    elif "wickets" in status:
        win_type = "wickets"

    cursor.execute("""
        UPDATE matches_v2
        SET win_type = %s
        WHERE match_id = %s
    """, (win_type, match_id))

In [441]:
def store_player_performance(cursor, data, match_id):

    for innings_data in data["scorecard"]:
        innings = innings_data["inningsid"]

        # -------- BATTING --------
        for b in innings_data.get("batsman", []):
            cursor.execute("""
                INSERT INTO player_match_performance
                (player_id, match_id, innings, runs, balls, strike_rate)
                VALUES (%s,%s,%s,%s,%s,%s)
            """, (
                b["id"],
                match_id,
                innings,
                b.get("runs", 0),
                b.get("balls", 0),
                float(b.get("strkrate", 0))
            ))

        # -------- BOWLING --------
        for b in innings_data.get("bowler", []):
            cursor.execute("""
                INSERT INTO player_match_performance
                (player_id, match_id, innings, overs, wickets, economy)
                VALUES (%s,%s,%s,%s,%s,%s)
            """, (
                b["id"],
                match_id,
                innings,
                float(b.get("overs", 0)),
                b.get("wickets", 0),
                float(b.get("economy", 0))
            ))

In [ ]:
def store_partnerships(cursor, data, match_id):

    for innings_data in data["scorecard"]:
        innings = innings_data["inningsid"]

        partnerships = innings_data.get("partnership", {}).get("partnership", [])

        for p in partnerships:
            cursor.execute("""
                INSERT INTO partnerships
                (match_id, innings, player1, player2, partnership_runs)
                VALUES (%s,%s,%s,%s,%s)
            """, (
                match_id,
                innings,
                p.get("bat1name"),   
                p.get("bat2name"),  
                p.get("totalruns", 0)
            ))

In [443]:
def get_match_ids(conn):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT match_id
        FROM matches_v2
        WHERE match_id IS NOT NULL
        ORDER BY match_id DESC
        LIMIT 20;
    """)
    rows = cursor.fetchall()
    return [r[0] for r in rows]

In [ ]:
def process_valid_matches(target_count=4):
    conn = get_connection()
    if conn is None:
        return

    cursor = conn.cursor()
    match_ids = get_match_ids(conn)

    success_count = 0

    for match_id in match_ids:
        print(f"\nTrying match {match_id}")

        data = fetch_match_data(match_id)

        if data is None:
            continue

        # 🔥 validation
        if "scorecard" not in data:
            print("No scorecard - skipping")
            continue

        try:
            store_match_info(cursor, data, match_id)
            store_player_performance(cursor, data, match_id)
            store_partnerships(cursor, data, match_id)

            conn.commit()
            print(f"SUCCESS match {match_id}")

            success_count += 1

        except Exception as e:
            conn.rollback()
            print(f"Error {match_id}: {e}")

        if success_count >= target_count:
            print("\nTarget reached!")
            break

        time.sleep(1.5)

    cursor.close()
    conn.close()

In [448]:
process_valid_matches()


🔄 Trying match 151158

🔍 RAW RESPONSE for 151158:
{'scorecard': [{'inningsid': 1, 'batsman': [{'id': 1451395, 'balls': 14, 'runs': 1, 'fours': 0, 'sixes': 0, 'strkrate': '7.14', 'name': 'TV Badri Narayanan', 'nickname': 'TV Badri Narayanan', 'iscaptain': False, 'iskeeper': True, 'outdec': 'b Luiz Gabriel', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 1470370, 'balls': 4, 'runs': 1, 'fours': 0, 'sixes': 0, 'strkrate': '25', 'name': 'Joel Cutinho', 'nickname': 'Joel Cutinho', 'iscaptain': False, 'iskeeper': False, 'outdec': 'run out (Michel Assuncao/Chrystian Machado)', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 1470381, 'balls': 41, 'runs': 17, '

In [ ]:
import http.client
import json
import time

def update_match_results():
    conn = get_connection()
    cursor = conn.cursor()

    # get match_ids from DB
    cursor.execute("""
        SELECT match_id 
        FROM matches_v2
        WHERE match_id IS NOT NULL
        LIMIT 20;
    """)
    
    match_ids = [row[0] for row in cursor.fetchall()]

    for match_id in match_ids:
        print(f"🔄 Updating match {match_id}")

        try:
            conn_http = http.client.HTTPSConnection(
                "cricbuzz-cricket.p.rapidapi.com", timeout=10
            )

            headers = {
                    'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
                    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
                'Content-Type': "application/json"
            }

            conn_http.request(
                "GET", f"/mcenter/v1/{match_id}/scard", headers=headers
            )

            res = conn_http.getresponse()

            if res.status != 200:
                print(f"Failed {match_id}")
                continue

            data = json.loads(res.read().decode("utf-8"))

            # 🔥 extract result safely
            result = data.get("matchHeader", {}).get("result")

            if not result:
                print(f"No result for {match_id}, skipping")
                continue

            win_margin = result.get("winningMargin")

            if result.get("winByRuns"):
                win_type = "runs"
            elif win_margin is not None:
                win_type = "wickets"
            else:
                win_type = None
            if win_margin is None:
                print(f"No win_margin for {match_id}, skipping")
                continue
            win_type = None

            if result.get("winByRuns"):
                win_type = "runs"
            elif result.get("winningMargin") is not None:
                win_type = "wickets"

            # update DB
            cursor.execute("""
                UPDATE matches_v2
                SET win_margin = %s,
                    win_type = %s
                WHERE match_id = %s
            """, (win_margin, win_type, match_id))

            conn.commit()
            print(f"Updated {match_id}")

            time.sleep(1.5)

        except Exception as e:
            conn.rollback()
            print(f"Error {match_id}: {e}")

    cursor.close()
    conn.close()

In [454]:
process_single_match(40381)
process_single_match(40378)

🔄 Processing match 40381

🔍 RAW RESPONSE for 40381:
{'scorecard': [{'inningsid': 1, 'batsman': [{'id': 1114, 'balls': 5, 'runs': 5, 'fours': 1, 'sixes': 0, 'strkrate': '100', 'name': 'Stirling', 'nickname': '', 'iscaptain': False, 'iskeeper': False, 'outdec': 'lbw b Nisarg Patel', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 6710, 'balls': 5, 'runs': 10, 'fours': 2, 'sixes': 0, 'strkrate': '200', 'name': 'Balbirnie', 'nickname': '', 'iscaptain': True, 'iskeeper': False, 'outdec': 'c Sushant Modani b Netravalkar', 'videotype': '', 'videourl': '', 'videoid': 0, 'planid': 0, 'imageid': 0, 'premiumvideourl': '', 'iscbplusfree': False, 'ispremiumfree': False, 'inmatchchange': '', 'isoverseas': False, 'playingxichange': ''}, {'id': 11131, 'balls': 56, 'runs': 84, 'fours': 9, 'sixes': 3, 'strkrate': '150', 'name': 'Lorcan T

Bowler stats question

In [455]:
import psycopg2

def fetch_players():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        SELECT player_id, player_name 
        FROM players
        WHERE bowling_style IS NOT NULL
    """)

    players = cur.fetchall()

    cur.close()
    conn.close()

    return players


players_list = fetch_players()

print("Total Players:", len(players_list))
players_list[:5]

Total Players: 486


[(11808, 'Shubman Gill'),
 (13940, 'Yashasvi Jaiswal'),
 (13866, 'Sai Sudharsan'),
 (576, 'Rohit Sharma'),
 (1413, 'Virat Kohli')]

In [456]:
import http.client
import json

def fetch_player_bowling(player_id):
    conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
            'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
            'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn_api.request("GET", f"/stats/v1/player/{player_id}/bowling", headers=headers)

    res = conn_api.getresponse()
    data = res.read()

    return json.loads(data.decode("utf-8"))

In [457]:
def extract_stat(values, stat_name):
    for item in values:
        if item["values"][0] == stat_name:
            return item["values"]
    return None

In [458]:
def compute_bowler_stats(player_id, player_name):
    try:
        data = fetch_player_bowling(player_id)

        values = data["values"]

        matches = extract_stat(values, "Matches")
        balls = extract_stat(values, "Balls")
        runs = extract_stat(values, "Runs")
        wickets = extract_stat(values, "Wickets")

        if not matches:
            return None

        # ODI = index 2, T20 = index 3
        odi_matches = int(matches[2])
        t20_matches = int(matches[3])

        odi_balls = int(balls[2])
        t20_balls = int(balls[3])

        odi_runs = int(runs[2])
        t20_runs = int(runs[3])

        odi_wickets = int(wickets[2])
        t20_wickets = int(wickets[3])

        total_matches = odi_matches + t20_matches
        total_balls = odi_balls + t20_balls
        total_runs = odi_runs + t20_runs
        total_wickets = odi_wickets + t20_wickets

        if total_matches == 0:
            return None

        avg_overs = (total_balls / 6) / total_matches

        if total_matches >= 10 and avg_overs >= 2:
            economy = total_runs / (total_balls / 6)

            return (
                player_id,
                player_name,
                total_matches,
                total_wickets,
                total_balls,
                total_runs,
                round(economy, 2)
            )

    except Exception as e:
        print(f"Error for player {player_id}: {e}")

    return None

In [459]:
def create_table():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS bowler_economy_stats (
        player_id BIGINT PRIMARY KEY,
        player_name TEXT,
        total_matches INT,
        total_wickets INT,
        total_balls INT,
        total_runs INT,
        economy NUMERIC
    );
    """)

    conn.commit()
    cur.close()
    conn.close()

create_table()

In [460]:
def insert_stats(row):
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO bowler_economy_stats 
        (player_id, player_name, total_matches, total_wickets, total_balls, total_runs, economy)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (player_id) DO UPDATE SET
            total_matches = EXCLUDED.total_matches,
            total_wickets = EXCLUDED.total_wickets,
            total_balls = EXCLUDED.total_balls,
            total_runs = EXCLUDED.total_runs,
            economy = EXCLUDED.economy;
    """, row)

    conn.commit()
    cur.close()
    conn.close()

In [ ]:
# LIMIT to avoid API rate limit
players_list = players_list[:50]

for player_id, player_name in players_list:
    stats = compute_bowler_stats(player_id, player_name)

    if stats:
        insert_stats(stats)


Error for player 9716: Expecting value: line 1 column 1 (char 0)
Error for player 14281: Expecting value: line 1 column 1 (char 0)
✅ Done


In [464]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/mcenter/v1/149684", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"matchid":149684,"seriesid":9241,"seriesname":"Indian Premier League 2026","matchdesc":"7th Match","matchformat":"T20","startdate":1775224800000,"enddate":1775238813201,"state":"Complete","status":"Punjab Kings opt to bowl","team1":{"teamid":58,"teamname":"Chennai Super Kings","teamsname":"CSK","isfullmember":false,"isassociated":false,"isleagueteam":false,"iswomenteam":false,"isheader":false,"isactive":false,"teampriority":"","isvideopresent":false,"imageid":0,"countryname":"","belongsto":"","teamcolor":""},"team2":{"teamid":65,"teamname":"Punjab Kings","teamsname":"PBKS","isfullmember":false,"isassociated":false,"isleagueteam":false,"iswomenteam":false,"isheader":false,"isactive":false,"teampriority":"","isvideopresent":false,"imageid":0,"countryname":"","belongsto":"","teamcolor":""},"umpire1":{"id":14178,"name":"Akshay Totre","country":"IND"},"umpire2":{"id":11521,"name":"Abhijit Bhattacharya","country":"IND"},"umpire3":{"id":8871,"name":"KN Anantha Padmanabhan","country":"IND"},"

In [ ]:
import http.client
import json

def fetch_match_details(match_id):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "YOUR_API_KEY",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    conn.request("GET", f"/mcenter/v1/{match_id}", headers=headers)

    res = conn.getresponse()
    data = res.read()

    return json.loads(data.decode("utf-8"))

In [466]:

def create_toss_table():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS match_toss (
            match_id BIGINT PRIMARY KEY,
            toss_winner_team_id BIGINT,
            toss_decision TEXT
        );
    """)

    conn.commit()
    cur.close()
    conn.close()

create_toss_table()

In [467]:
import http.client
import json

def fetch_match_details(match_id):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request("GET", f"/mcenter/v1/{match_id}", headers=headers)

    res = conn.getresponse()
    data = res.read()

    return json.loads(data.decode("utf-8"))

In [468]:
def parse_toss(toss_status):
    if not toss_status:
        return None, None

    toss_status = toss_status.lower()

    if "opt to" in toss_status:
        parts = toss_status.split(" opt to ")
    elif "elected to" in toss_status:
        parts = toss_status.split(" elected to ")
    else:
        return None, None

    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()

    return None, None

In [469]:
def get_pending_matches(limit=50):
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        SELECT m.match_id
        FROM matches_v2 m
        LEFT JOIN match_toss mt 
            ON m.match_id = mt.match_id
        WHERE mt.match_id IS NULL
        LIMIT %s
    """, (limit,))

    matches = cur.fetchall()

    cur.close()
    conn.close()

    return [m[0] for m in matches]

In [470]:
def insert_toss(match_id, toss_winner_id, decision):
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO match_toss (match_id, toss_winner_team_id, toss_decision)
        VALUES (%s, %s, %s)
        ON CONFLICT (match_id) DO NOTHING;
    """, (match_id, toss_winner_id, decision))

    conn.commit()
    cur.close()
    conn.close()

In [ ]:
def backfill_toss_data(limit=50):
    match_ids = get_pending_matches(limit)

    print(f"Processing {len(match_ids)} matches...")

    for match_id in match_ids:
        try:
            data = fetch_match_details(match_id)

            toss_status = data.get("tossstatus", "")
            toss_winner_name, decision = parse_toss(toss_status)

            if not toss_winner_name or not decision:
                continue

            # Team info from API
            t1_name = data["team1"]["teamname"]
            t1_id = data["team1"]["teamid"]

            t2_name = data["team2"]["teamname"]
            t2_id = data["team2"]["teamid"]

            # Map toss winner → team_id
            if toss_winner_name.lower() in t1_name.lower():
                toss_winner_id = t1_id
            elif toss_winner_name.lower() in t2_name.lower():
                toss_winner_id = t2_id
            else:
                continue

            insert_toss(match_id, toss_winner_id, decision)

            print(f"Stored toss for match {match_id}")

        except Exception as e:
            print(f"Error for match {match_id}: {e}")

    print("Backfill completed")

In [472]:
backfill_toss_data(limit=50)

Processing 41 matches...
✅ Stored toss for match 148973
✅ Stored toss for match 147477
✅ Stored toss for match 133446
✅ Stored toss for match 132357
✅ Stored toss for match 132368
✅ Stored toss for match 132373
✅ Stored toss for match 122858
✅ Stored toss for match 122847
✅ Stored toss for match 122836
✅ Stored toss for match 148375
✅ Stored toss for match 148364
✅ Stored toss for match 150175
✅ Stored toss for match 150164
✅ Stored toss for match 152296
✅ Stored toss for match 151147
✅ Stored toss for match 151141
✅ Stored toss for match 152302
✅ Stored toss for match 151169
✅ Stored toss for match 151158
✅ Stored toss for match 148265
🎯 Backfill completed


In [473]:
import http.client

conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "95eaea79e0mshc9bd613262acc8ep1acc08jsn4370aaa63480",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    'Content-Type': "application/json"
}

conn.request("GET", "/stats/v1/player/8733/batting", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"message":"You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https:\/\/rapidapi.com\/cricketapilive\/api\/cricbuzz-cricket"}


In [15]:
import http.client
import json
import time
import psycopg2
import pandas as pd

#### Get Match ID's from DB

In [63]:
def get_match_ids(limit=20):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        SELECT match_id 
        FROM matches_v2 
        ORDER BY start_date DESC 
        LIMIT %s;
    """

    cursor.execute(query, (limit,))
    rows = cursor.fetchall()

    cursor.close()
    conn.close()

    return [row[0] for row in rows]

#### Extract Partnerships from API

In [12]:
def extract_partnerships(match_ids):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    results = []

    for match_id in match_ids:
        try:
            conn.request("GET", f"/mcenter/v1/{match_id}/hscard", headers=headers)
            res = conn.getresponse()
            data = res.read()

            if not data:
                continue

            match_data = json.loads(data.decode("utf-8"))

            for innings in match_data.get("scorecard", []):
                innings_id = innings.get("inningsid")

                partnerships = innings.get("partnership", {}).get("partnership", [])

                for p in partnerships:
                    results.append({
                        "match_id": match_id,
                        "innings": innings_id,
                        "player1": p.get("bat1name"),
                        "player2": p.get("bat2name"),
                        "partnership_runs": p.get("totalruns", 0)
                    })

        except Exception as e:
            print(f"Error in match {match_id}: {e}")

        time.sleep(1)  # avoid API rate limit

    return results

#### Insert Into Database

In [13]:
def insert_partnerships(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO partnerships (
            match_id, innings, player1, player2, partnership_runs
        )
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING;
    """

    try:
        for row in data:
            cursor.execute(query, (
                row["match_id"],
                row["innings"],
                row["player1"],
                row["player2"],
                row["partnership_runs"]
            ))

        conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"Insertion error: {e}")

    finally:
        cursor.close()
        conn.close()

#### Run Pipeline

In [64]:
match_ids = get_match_ids(10)

print("Match IDs:", match_ids)

Match IDs: [122858, 151169, 151158, 149053, 149684, 152302, 152296, 151147, 148375, 151141]


In [16]:


data = extract_partnerships(match_ids)

df = pd.DataFrame(data)
df.head()

Match IDs: [122858, 151169, 151158, 149053, 149684, 152302, 152296, 151147, 148375, 151141]


,match_id,innings,player1,player2,partnership_runs
0,122858,1,Bates,Plimmer,3
1,122858,1,Amelia Kerr,Plimmer,0
2,122858,1,Amelia Kerr,Green,0
3,122858,1,Halliday,Green,211
4,122858,1,Isabella Gaze,Green,19


#### Insert Data

In [ ]:
insert_partnerships(data)

print("Data inserted successfully")

✅ Data inserted successfully


#### Verify

In [18]:
conn = get_connection()

query = """
SELECT *
FROM partnerships
ORDER BY partnership_runs DESC
LIMIT 10;
"""

df = pd.read_sql(query, conn)
conn.close()

df

C:\Users\hp\AppData\Local\Temp\ipykernel_14448\3230183502.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,match_id,innings,player1,player2,partnership_runs,created_at
0,220,122858,1,Halliday,Green,211,2026-04-21 11:02:36.227450
1,329,151147,1,Amir Butt,Dhananjaya Panda,135,2026-04-21 11:02:36.227450
2,164,151147,1,Amir Butt,Dhananjaya Panda,135,2026-04-05 16:20:47.453526
3,207,40378,1,Sushant Modani,Gajanand,110,2026-04-05 19:14:29.386386
4,29,40378,1,Sushant Modani,Gajanand,110,2026-04-05 15:21:51.476566
5,278,149684,1,Ayush Mhatre,Ruturaj Gaikwad,96,2026-04-21 11:02:36.227450
6,348,148375,2,Perry,Phoebe Litchfield,87,2026-04-21 11:02:36.227450
7,265,149053,1,Mohammad Naeem,Parvez Hossain Emon,86,2026-04-21 11:02:36.227450
8,181,151141,2,Yasar Haroon,Luiz Morais,81,2026-04-05 16:20:49.845142
9,357,151141,2,Yasar Haroon,Luiz Morais,81,2026-04-21 11:02:36.227450


#### Bowling Performance Info Inserting

In [50]:
import http.client
import json

def get_match_ids(limit=20):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    conn.request("GET", "/matches/v1/recent", headers=headers)

    res = conn.getresponse()
    data = res.read()

    matches_json = json.loads(data.decode("utf-8"))

    match_ids = []

    try:
        for type_match in matches_json.get("typeMatches", []):
            for series in type_match.get("seriesMatches", []):

                # skip ads
                if "seriesAdWrapper" not in series:
                    continue

                matches = series["seriesAdWrapper"].get("matches", [])

                for match in matches:
                    match_id = match.get("matchInfo", {}).get("matchId")

                    if match_id:
                        match_ids.append(match_id)

                    if len(match_ids) >= limit:
                        return match_ids

    except Exception as e:
        print("Error extracting match_ids:", e)

    return match_ids

In [51]:
match_ids = get_match_ids(10)
print(match_ids)

[150098, 150087, 153794, 148903, 148892, 148874, 151845, 151840, 151829, 151818]


In [52]:
def extract_bowling(match_ids):
    import http.client
    import json
    import time

    http_conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    results = []

    for match_id in match_ids:
        try:
            http_conn.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
            res = http_conn.getresponse()
            data = res.read()

            match_data = json.loads(data.decode("utf-8"))

            for innings in match_data.get("scorecard", []):

                bowlers = innings.get("bowler", [])  # ✅ CORRECT KEY

                for b in bowlers:
                    results.append({
                        "match_id": match_id,
                        "player_id": b.get("id"),
                        "player_name": b.get("name"),
                        "overs": float(b.get("overs", 0)),
                        "runs": int(b.get("runs", 0)),
                        "wickets": int(b.get("wickets", 0)),
                        "economy": float(b.get("economy", 0))
                    })

        except Exception as e:
            print(f"Error in match {match_id}: {e}")

        time.sleep(1)

    return results

In [53]:
data = extract_bowling([151829])
print(len(data))
print(data[:2])

13
[{'match_id': 151829, 'player_id': 15861, 'player_name': 'Vaibhav Arora', 'overs': 4.0, 'runs': 37, 'wickets': 0, 'economy': 9.2}, {'match_id': 151829, 'player_id': 13136, 'player_name': 'Kartik Tyagi', 'overs': 4.0, 'runs': 22, 'wickets': 3, 'economy': 5.5}]


In [54]:
import http.client
import json

http_conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

headers = {
    'x-rapidapi-key': "4172f835b0mshb6d6a65d2c10591p19abbajsn4d930e083f39",
    'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
}

http_conn.request("GET", "/mcenter/v1/151829/scard", headers=headers)

res = http_conn.getresponse()
data = res.read()

match_data = json.loads(data.decode("utf-8"))

print(match_data.keys())
print(match_data.get("scorecard")[0].keys())

dict_keys(['scorecard', 'ismatchcomplete', 'appindex', 'status', 'responselastupdated'])
dict_keys(['inningsid', 'batsman', 'bowler', 'fow', 'extras', 'pp', 'score', 'wickets', 'overs', 'runrate', 'batteamname', 'batteamsname', 'isdeclared', 'isfollowon', 'ballnbr', 'rpb', 'partnership'])


In [42]:
extract_bowling(match_ids=[151829])

[]

#### Insert Into DB

In [56]:
def insert_bowling(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO bowling_performance_v2 (
            match_id, player_id, player_name, overs, runs, wickets, economy
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING;
    """

    for row in data:
        cursor.execute(query, (
            row["match_id"],
            row["player_id"],
            row["player_name"],
            row["overs"],
            row["runs"],
            row["wickets"],
            row["economy"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [57]:
match_ids = get_match_ids(20)

data = extract_bowling(match_ids)

print("Extracted:", len(data))

insert_bowling(data)

Extracted: 92


In [25]:
print(len(match_ids))

20


#### Team Mapping

In [97]:
def get_team_mapping():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT team_id, team_name FROM teams;")
    rows = cursor.fetchall()

    cursor.close()
    conn.close()

    team_map = {}

    for team_id, name in rows:
        key = name.lower().strip().replace(" ", "")
        team_map[key] = team_id

    return team_map

In [98]:
def get_match_ids():
    return [
        151829, 46046, 44271, 44291, 44276,
        44281, 44286, 44631, 44616, 44621
    ]

#### Extract Player Performance

In [ ]:
def extract_player_performance(match_ids):
    import http.client, json, time

    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "6c706987fcmsha3b8ea8bab5b3aap148993jsna65beff53f0d",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    results = []

    for match_id in match_ids:
        try:
            conn.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
            res = conn.getresponse()
            data = res.read()

            match_data = json.loads(data.decode("utf-8"))

            for innings in match_data.get("scorecard", []):

                team_name = innings.get("batteamname")

       
                for b in innings.get("batsman", []):

                    results.append({
                        "match_id": match_id,
                        "player_id": b.get("id"),
                        "runs": b.get("runs", 0),
                        "team_name": team_name
                    })

        except Exception as e:
            print(f"Error in match {match_id}: {e}")

        time.sleep(1)

    return results

#### Insert into DB

In [124]:
def insert_player_performance(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO player_match_performance_v2 (
            match_id, player_id, runs, team_id
        )
        VALUES (%s, %s, %s,
            (SELECT team_id FROM teams WHERE LOWER(team_name) = LOWER(%s) LIMIT 1)
        )
        ON CONFLICT DO NOTHING;
    """

    for row in data:
        cursor.execute(query, (
            row["match_id"],
            row["player_id"],
            row["runs"],
            row["team_name"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [125]:
data = extract_player_performance(match_ids)

print("Extracted:", len(data))

insert_player_performance(data)

Extracted: 346


In [ ]:
def get_matches_data(limit=15):
    import http.client, json

    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "6c706987fcmsha3b8ea8bab5b3aap148993jsna65beff53f0d",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com",
    }

    conn.request("GET", "/matches/v1/recent", headers=headers)
    res = conn.getresponse()
    data = json.loads(res.read().decode("utf-8"))

    results = []

    for type_match in data.get("typeMatches", []):
        for series in type_match.get("seriesMatches", []):
            wrapper = series.get("seriesAdWrapper")

            if not wrapper:
                continue

            for match in wrapper.get("matches", []):
                info = match.get("matchInfo", {})

                if info.get("state") != "Complete":
                    continue

                match_id = info.get("matchId")
                team1_id = info.get("team1", {}).get("teamId")
                team2_id = info.get("team2", {}).get("teamId")
                status = info.get("status", "").lower()

                win_type = None
                win_margin = None
                winner_team_id = None

                if "won by" in status:
                    if "runs" in status:
                        win_type = "runs"
                        win_margin = int(status.split("won by")[1].split("runs")[0].strip())
                    elif "wkt" in status:
                        win_type = "wickets"
                        win_margin = int(status.split("won by")[1].split("wkt")[0].strip())

                    if info.get("team1", {}).get("teamName", "").lower() in status:
                        winner_team_id = team1_id
                    else:
                        winner_team_id = team2_id

                results.append({
                    "match_id": match_id,
                    "team1_id": team1_id,
                    "team2_id": team2_id,
                    "winner_team_id": winner_team_id,
                    "win_type": win_type,
                    "win_margin": win_margin
                })

                if len(results) >= limit:
                    return results

    return results

In [117]:
def insert_matches(data):
    conn = get_connection()
    cursor = conn.cursor()

    query = """
        INSERT INTO matches_v2 (
            match_id, team1_id, team2_id,
            winner_team_id, win_type, win_margin
        )
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING;
    """

    for row in data:
        cursor.execute(query, (
            row["match_id"],
            row["team1_id"],
            row["team2_id"],
            row["winner_team_id"],
            row["win_type"],
            row["win_margin"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [118]:
matches = get_matches_data(15)

print("Matches:", len(matches))

insert_matches(matches)

match_ids = [m["match_id"] for m in matches]

data = extract_player_performance(match_ids)

print("Player rows:", len(data))

insert_player_performance(data)

Matches: 15
Player rows: 340


#### Get Match ID's

In [151]:
def get_match_ids(limit=30):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "2767392619mshf3f20e34f8b3117p1ddd4fjsnb30bd89e03ed",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    conn.request("GET", "/matches/v1/recent", headers=headers)
    res = conn.getresponse()
    data = json.loads(res.read().decode("utf-8"))

    match_ids = []

    for type_match in data.get("typeMatches", []):
        for series_block in type_match.get("seriesMatches", []):

            wrapper = series_block.get("seriesAdWrapper")
            if not wrapper:
                continue

            for match in wrapper.get("matches", []):
                match_ids.append(match["matchInfo"]["matchId"])

                if len(match_ids) >= limit:
                    return match_ids

    return match_ids

In [ ]:
def get_valid_match_ids(match_ids):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "2767392619mshf3f20e34f8b3117p1ddd4fjsnb30bd89e03ed",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    valid_ids = []

    for match_id in match_ids:
        try:
            conn.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
            res = conn.getresponse()
            data = json.loads(res.read().decode("utf-8"))

            if data.get("scorecard"):  
                valid_ids.append(match_id)

        except:
            continue

    return valid_ids

#### Insert Matches

In [ ]:
def insert_matches(match_ids):
    conn_api = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "2767392619mshf3f20e34f8b3117p1ddd4fjsnb30bd89e03ed",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    conn_db = get_connection()
    cursor = conn_db.cursor()

    for match_id in match_ids:
        try:
            conn_api.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
            res = conn_api.getresponse()
            data = json.loads(res.read().decode("utf-8"))

            scorecard = data.get("scorecard", [])

            if not scorecard:
                print("No scorecard:", match_id)
                continue

            # Extract team names from innings
            team1 = scorecard[0].get("batteamname")
            team2 = scorecard[1].get("batteamname") if len(scorecard) > 1 else None

            # Since we don't have team_id mapping, keep NULL for now
            cursor.execute("""
                INSERT INTO matches_v2 (
                    match_id,
                    match_format,
                    start_date,
                    team1_id,
                    team2_id,
                    winner_team_id
                )
                VALUES (%s, %s, NOW(), NULL, NULL, NULL)
                ON CONFLICT (match_id) DO NOTHING;
            """, (
                match_id,
                "T20" 
            ))

        except Exception as e:
            print("Match insert error:", match_id, e)

    conn_db.commit()
    cursor.close()
    conn_db.close()

In [154]:
def extract_player_performance(match_ids):
    conn = http.client.HTTPSConnection("cricbuzz-cricket.p.rapidapi.com")

    headers = {
        'x-rapidapi-key': "2767392619mshf3f20e34f8b3117p1ddd4fjsnb30bd89e03ed",
        'x-rapidapi-host': "cricbuzz-cricket.p.rapidapi.com"
    }

    data_list = []

    for match_id in match_ids:
        try:
            conn.request("GET", f"/mcenter/v1/{match_id}/scard", headers=headers)
            res = conn.getresponse()
            data = json.loads(res.read().decode("utf-8"))

            scorecard = data.get("scorecard", [])

            if not scorecard:
                print("No scorecard for match", match_id)
                continue

            for innings in scorecard:
                for bat in innings.get("batsman", []):
                    if not bat.get("id"):
                        continue

                    data_list.append({
                        "match_id": match_id,
                        "player_id": bat.get("id"),
                        "runs": bat.get("runs")
                    })

        except Exception as e:
            print("Error:", match_id, e)

    return data_list

In [ ]:
def insert_player_performance(data):
    conn = get_connection()  
    cursor = conn.cursor()

    query = """
        INSERT INTO player_match_performance_v2 (
            match_id, player_id, runs
        )
        VALUES (%s, %s, %s)
        ON CONFLICT DO NOTHING;
    """

    for row in data:
        cursor.execute(query, (
            row["match_id"],
            row["player_id"],
            row["runs"]
        ))

    conn.commit()
    cursor.close()
    conn.close()

In [156]:
match_ids = get_match_ids(30)

print("All Match IDs:", match_ids)

valid_match_ids = get_valid_match_ids(match_ids)

print("Valid Match IDs:", valid_match_ids)

insert_matches(valid_match_ids)

data = extract_player_performance(valid_match_ids)

print("Extracted:", len(data))

insert_player_performance(data)

All Match IDs: [150098, 150087, 153803, 153794, 148903, 148892, 148874, 151856, 151845, 151840, 151829, 151818, 151807, 151796, 151785, 151774, 149260, 149249, 149238, 149227, 149216, 149215, 149204, 149193, 149182, 142131, 142142, 150327, 142825, 142836]
Valid Match IDs: [150098, 150087, 153803, 153794, 148903, 148892, 148874, 151856, 151845, 151840, 151829, 151818, 151807, 151796, 151785, 151774, 149260, 149249, 149238, 149227, 149216, 149215, 149204, 149193, 149182, 142131, 142142, 150327, 142825, 142836]
Extracted: 764
